[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/080.%20The%20International%20Freight%20Mode%20Selection%20Problem%20%28Air%20vs.%20Ocean%29/P80-Tier-8_executed.ipynb)

---


# 80. The International Freight Mode Selection Problem (Air vs. Ocean)
## Tier 8: Value-Aligned and Ethical Decision Framework

### Purpose and scope

This notebook demonstrates a value-aligned freight-mode decision for a set of shipments. For each shipment, the model evaluates **ocean** and **air** transport, excludes any mode that violates a hard ethical constraint when a feasible mode exists, and selects the remaining mode with the highest weighted objective score. It then reports economic, environmental, social, and compliance measures and presents stakeholder-oriented summaries.

The inputs are shipment characteristics, mode-specific cost, transit-time, emissions, and energy data, stakeholder definitions, and ethical constraints. The outputs are one mode decision per shipment, a transparent decision report, summary metrics, and visualizations. The method is a finite comparison of two alternatives using a weighted score and explicit hard-constraint screening; it is not a general-purpose multi-objective solver.

### Model assumptions

- Each shipment is considered independently; there is no shared capacity allocation, route planning, or interaction between shipments. The supplied air-capacity parameter is recorded but is not enforced by this per-shipment decision model.
- Transportation rates, transit times, emissions intensities, and energy intensities are deterministic and constant for each mode-destination pair.
- The economic cost includes transport cost, the stated holding-cost convention, late penalties, and specified temperature-control or hazardous-cargo charges. Holding cost is charged for the time between arrival and the due date when the shipment arrives early; it is not a general inventory-cost model.
- Environmental and social scores are simplified, normalized indicators based on the assumptions in the code. They are illustrative model inputs, not audited life-cycle assessments or evidence about actual carriers.
- Hard constraints determine feasibility. Soft constraints contribute a penalty to the weighted objective, while guidance principles are reported as compliance checks but carry no violation penalty. The fallback used when both alternatives violate hard constraints selects the mode with fewer hard-constraint violations; it does not make that mode compliant.
- Stakeholder utilities are descriptive components of the reported ethical score. The selected mode is determined by the optimizer's stated weighted economic, environmental, and social objective, with constraint penalties—not by maximizing stakeholder utilities directly.
- The example does not model labor-provider data, measured noise in decibels, regulatory jurisdictions, uncertainty, or stakeholder deliberation. Where the illustrative data use normalized proxies, the units and limitations are stated in the code and reports.

### Decision sequence

1. Define shipments, transport data, stakeholders, and ethical constraints.
2. Calculate economic, environmental, and social measures for each shipment-mode pair.
3. Check compliance and remove modes that violate hard constraints when a feasible mode is available.
4. Rank remaining alternatives using the weighted objective and choose a mode.
5. Record the decision, report the alternative, and summarize the results.

### Mathematical formulation

Let \(I\) be the set of shipments and \(M=\{\text{ocean},\text{air}\}\) the available modes. For shipment \(i\in I\) and mode \(m\in M\), let \(C_{im}\) be total economic cost; let \(E_{im}\in[0,1]\) and \(S_{im}\in[0,1]\) be the normalized environmental and social scores, with larger scores representing better performance; and let \(H_{im}\) be the number of violated hard constraints. Let \(P_{im}\) be the weighted penalty for violated soft constraints. The mode evaluation score is

$$
Q_{im}=w_C N(C_{im})+w_E E_{im}+w_S S_{im}-0.1P_{im},
\qquad w_C=0.30,\quad w_E=0.35,\quad w_S=0.35,
$$

where \(N(C_{im})\in[0,1]\) is the normalized cost score, for which higher is better. Define the feasible set \(F_i=\{m\in M:H_{im}=0\}\). If \(F_i\neq\varnothing\), the selected mode maximizes \(Q_{im}\) over \(m\in F_i\). If \(F_i=\varnothing\), the fallback first minimizes \(H_{im}\); ties are resolved by the objective score, with ocean first when scores are equal within floating-point tolerance.

The hard-constraint count, soft-constraint penalty, and objective score are implemented by `ValueAlignedOptimizer._count_hard_violations`, `EthicalConstraintChecker.get_violation_penalty`, and `ValueAlignedOptimizer._evaluate_mode`. The selection and fallback are implemented by `optimize_freight_mode` and `_select_best_option`. This is a two-alternative evaluation for each shipment, so the number of mode evaluations grows linearly with the number of shipments; it does not search joint assignments across shipments.

For each mode, the economic cost is calculated as

$$
C_{im}=w_i c_{md_i}+h_i\max(0,d_i-t_{md_i})+p\max(0,t_{md_i}-d_i)+K_{im},
$$

where \(w_i\) is shipment weight, \(c_{md_i}\) is the mode-destination transport cost per kilogram, \(h_i=r v_i\) is the daily holding cost based on value \(v_i\) and holding rate \(r\), \(d_i\) is the due date in days, \(t_{md_i}\) is transit time, \(p\) is the late penalty per day, and \(K_{im}\) is the specified special-requirements charge. The corresponding code is `ValueAlignedOptimizer._calculate_total_cost`.

### How to interpret the results

A selected mode is the best-scoring available mode under these illustrative weights and constraints, not a universally best freight choice. A high compliance percentage is a simple fraction of checks passed; it is not a measure of the severity of violations or proof of ethical performance. Comparisons and correlations across different shipments are descriptive because shipments have different weights, values, destinations, and due dates.

In [ ]:
import math
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 1. Define the decision data

The data classes make the model inputs explicit. A shipment contains its weight, value, destination, due date, priority, special handling requirements, and shipment-specific stakeholder-importance weights. `TransportParams` stores mode-destination rates and impact intensities. Stakeholders and ethical constraints are represented separately so their definitions can be inspected and passed to the scoring and checking components.

The numeric values below are illustrative inputs. In particular, the labor, community, and noise indicators used later are proxies; the example does not provide carrier-level measurements.

In [ ]:
@dataclass
class Shipment:
    """Shipment data and shipment-specific stakeholder importance."""
    id: int
    weight: float  # kg
    value: float  # USD
    destination: str
    due_date: int  # days from the decision date
    priority: str
    special_requirements: List[str]
    stakeholder_importance: Dict[str, float]


@dataclass
class TransportParams:
    """Mode-destination transport and impact parameters."""
    costs: Dict[str, Dict[str, float]]
    transit_times: Dict[str, Dict[str, int]]
    holding_rate: float
    late_penalty: float
    air_capacity: float  # kg; not enforced in this independent-shipment model
    carbon_emissions: Dict[str, Dict[str, float]]  # kg CO2 per kg cargo
    energy_consumption: Dict[str, Dict[str, float]]  # kWh per kg cargo


@dataclass
class Stakeholder:
    """Stakeholder interests, utility weights, and influence attributes."""
    stakeholder_id: str
    name: str
    stakeholder_type: str
    interests: List[str]
    utility_weights: Dict[str, float]
    veto_power: bool
    influence_level: float


@dataclass
class EthicalConstraint:
    """A threshold check, classified as hard, soft, or guidance."""
    constraint_id: str
    name: str
    constraint_type: str
    description: str
    metric: str
    threshold: float
    penalty_weight: float
    applicable_destinations: List[str]


@dataclass
class EthicalDecision:
    """Selected mode and the measures used to explain the decision."""
    shipment_id: int
    chosen_mode: str
    economic_cost: float
    environmental_impact: Dict[str, float]
    social_impact: Dict[str, float]
    stakeholder_utilities: Dict[str, float]
    ethical_compliance: Dict[str, bool]
    ethical_score: float
    transparency_report: str
    alternative_considered: List[Dict]

In [ ]:
def create_ethical_framework_environment() -> Tuple[List[Shipment], TransportParams, List[Stakeholder], List[EthicalConstraint]]:
    """Create the illustrative shipment, transport, stakeholder, and constraint data."""
    shipments = [
        Shipment(1, 800, 1500000, "rotterdam", 8, "high", ["temperature_controlled"],
                 {"customer": 0.4, "environment": 0.3, "employee": 0.2, "shareholder": 0.1}),
        Shipment(2, 1200, 800000, "los_angeles", 15, "medium", [],
                 {"customer": 0.5, "shareholder": 0.3, "environment": 0.2}),
        Shipment(3, 3500, 600000, "rotterdam", 25, "low", ["hazardous"],
                 {"community": 0.4, "environment": 0.4, "regulator": 0.2}),
        Shipment(4, 600, 2000000, "singapore", 10, "high",
                 ["temperature_controlled", "pharmaceutical"],
                 {"customer": 0.6, "environment": 0.2, "employee": 0.2}),
        Shipment(5, 2000, 400000, "hamburg", 20, "medium", [],
                 {"shareholder": 0.5, "customer": 0.3, "environment": 0.2}),
        Shipment(6, 1500, 900000, "new_york", 12, "high", [],
                 {"customer": 0.4, "shareholder": 0.3, "environment": 0.3}),
        Shipment(7, 4000, 300000, "rotterdam", 35, "low", ["bulk_cargo"],
                 {"community": 0.3, "environment": 0.4, "shareholder": 0.3}),
        Shipment(8, 900, 1200000, "los_angeles", 18, "medium", ["fragile"],
                 {"customer": 0.5, "employee": 0.3, "environment": 0.2}),
        Shipment(9, 750, 1800000, "singapore", 7, "high", ["temperature_controlled"],
                 {"customer": 0.5, "environment": 0.3, "regulator": 0.2}),
        Shipment(10, 2800, 750000, "hamburg", 22, "medium", ["oversized"],
                 {"community": 0.4, "environment": 0.3, "shareholder": 0.3}),
    ]

    params = TransportParams(
        costs={
            "ocean": {"rotterdam": 2.2, "los_angeles": 1.9, "singapore": 2.8,
                      "hamburg": 2.1, "new_york": 1.8},
            "air": {"rotterdam": 6.8, "los_angeles": 6.2, "singapore": 7.5,
                    "hamburg": 6.5, "new_york": 5.9},
        },
        transit_times={
            "ocean": {"rotterdam": 28, "los_angeles": 20, "singapore": 35,
                      "hamburg": 26, "new_york": 18},
            "air": {"rotterdam": 3, "los_angeles": 2, "singapore": 4,
                    "hamburg": 3, "new_york": 2},
        },
        holding_rate=0.0012,
        late_penalty=1500,
        air_capacity=15000,
        carbon_emissions={
            "ocean": {"rotterdam": 0.015, "los_angeles": 0.012, "singapore": 0.018,
                      "hamburg": 0.014, "new_york": 0.011},
            "air": {"rotterdam": 0.85, "los_angeles": 0.78, "singapore": 0.92,
                    "hamburg": 0.82, "new_york": 0.75},
        },
        energy_consumption={
            "ocean": {"rotterdam": 0.08, "los_angeles": 0.07, "singapore": 0.09,
                      "hamburg": 0.08, "new_york": 0.06},
            "air": {"rotterdam": 2.4, "los_angeles": 2.2, "singapore": 2.6,
                    "hamburg": 2.3, "new_york": 2.1},
        },
    )

    stakeholders = [
        Stakeholder("STK001", "Customers", "customer",
                    ["cost_minimization", "service_quality", "reliability"],
                    {"cost": 0.4, "service": 0.4, "reliability": 0.2}, False, 0.3),
        Stakeholder("STK002", "Employees", "employee",
                    ["fair_wages", "working_conditions", "job_security"],
                    {"labor_standards": 0.5, "safety": 0.3, "job_security": 0.2}, False, 0.2),
        Stakeholder("STK003", "Local Communities", "community",
                    ["environmental_protection", "economic_development", "noise_reduction"],
                    {"environment": 0.4, "local_jobs": 0.3, "noise": 0.3}, False, 0.15),
        Stakeholder("STK004", "Environment", "environment",
                    ["carbon_reduction", "biodiversity_protection", "sustainability"],
                    {"carbon_emissions": 0.5, "energy_efficiency": 0.3,
                     "waste_reduction": 0.2}, False, 0.25),
        Stakeholder("STK005", "Shareholders", "shareholder",
                    ["profit_maximization", "risk_management", "long_term_growth"],
                    {"profit": 0.5, "risk": 0.3, "growth": 0.2}, False, 0.1),
        Stakeholder("STK006", "Regulators", "regulator",
                    ["compliance", "safety_standards", "environmental_regulations"],
                    {"compliance": 0.6, "safety": 0.3, "environment": 0.1}, True, 0.0),
    ]

    ethical_constraints = [
        EthicalConstraint("ETH001", "Carbon Emission Limit", "hard_constraint",
                          "Maximum CO2 emissions per shipment to meet climate goals",
                          "carbon_emissions", 1000.0, 1000.0, []),
        EthicalConstraint("ETH002", "Fair Labor Standards", "hard_constraint",
                          "Minimum labor standards for transportation providers",
                          "labor_standards", 0.8, 500.0, []),
        EthicalConstraint("ETH003", "Noise Pollution Control", "soft_constraint",
                          "Limit noise pollution in residential areas",
                          "noise_pollution", 70.0, 50.0, ["hamburg", "rotterdam"]),
        EthicalConstraint("ETH004", "Energy Efficiency", "guidance_principle",
                          "Prefer energy-efficient transportation modes",
                          "energy_efficiency", 2.0, 10.0, []),
        EthicalConstraint("ETH005", "Community Impact", "soft_constraint",
                          "Minimize negative impact on local communities",
                          "community_impact", 0.3, 30.0, ["los_angeles", "new_york"]),
    ]

    return shipments, params, stakeholders, ethical_constraints


shipments, params, stakeholders, ethical_constraints = create_ethical_framework_environment()

print("Illustrative decision environment initialized:")
print(f"  Shipments: {len(shipments)}")
print(f"  Stakeholders: {len(stakeholders)}")
print(f"  Ethical constraints: {len(ethical_constraints)}")
print("  Modes evaluated: ocean and air")

## 2. Score environmental, social, and stakeholder impacts

The scoring component translates unlike quantities into bounded indicators so the decision model can combine them. For a measure where lower values are preferable, the implementation uses a baseline divided by the observed value and clips the result to the interval \([0,1]\). Thus, a score is a relative indicator tied to the chosen baseline, not an absolute ethical rating.

For shipment \(i\), carbon emissions are cargo weight multiplied by the mode-destination emissions intensity. Energy consumption is calculated in the same way. The overall environmental score is the mean of the normalized carbon and energy scores. The social score is a simplified combination of labor, job-creation, working-condition, community, and noise proxies. Stakeholder utilities are weighted summaries of the interests that the scoring function explicitly maps; interests with no implemented mapping contribute zero. These utilities inform the reported ethical score and do not directly choose the mode.

The code retains explicit methods for each calculation so the assumptions and data flow can be inspected.

In [ ]:
class EthicalScoringSystem:
    """Calculate normalized impacts and illustrative stakeholder utilities."""

    def __init__(self, params: TransportParams, stakeholders: List[Stakeholder]):
        self.params = params
        self.stakeholders = stakeholders
        self.stakeholder_map = {stakeholder.stakeholder_id: stakeholder
                                for stakeholder in stakeholders}
        self.carbon_impact_weight = 0.3
        self.energy_impact_weight = 0.2
        self.social_impact_weight = 0.3
        self.economic_impact_weight = 0.2
        self.carbon_baseline = 500.0
        self.energy_baseline = 1.0
        self.economic_baseline = 1000.0

    def _normalize_impact(self, value: float, baseline: float,
                          lower_is_better: bool) -> float:
        """Map an illustrative impact to [0, 1], with larger scores preferred."""
        if lower_is_better:
            return max(0.0, min(1.0, baseline / max(value, 0.1)))
        return max(0.0, min(1.0, value / baseline))

    def calculate_environmental_impact(self, shipment: Shipment,
                                       mode: str) -> Dict[str, float]:
        """Calculate mode-specific emissions, energy, and normalized scores."""
        destination = shipment.destination
        carbon_emissions = (
            shipment.weight * self.params.carbon_emissions[mode][destination]
        )
        energy_consumption = (
            shipment.weight * self.params.energy_consumption[mode][destination]
        )
        carbon_score = self._normalize_impact(
            carbon_emissions, self.carbon_baseline, True
        )
        energy_score = self._normalize_impact(
            energy_consumption, self.energy_baseline * shipment.weight, True
        )

        if mode == "ocean":
            ballast_water_impact = 0.1 if shipment.weight > 2000 else 0.05
            marine_life_risk = (
                0.08 if "hazardous" in shipment.special_requirements else 0.02
            )
            air_pollution = 0.02
        else:
            ballast_water_impact = 0.0
            marine_life_risk = 0.0
            air_pollution = 0.15

        return {
            "carbon_emissions": carbon_emissions,
            "carbon_score": carbon_score,
            "energy_consumption": energy_consumption,
            "energy_score": energy_score,
            "air_pollution": air_pollution,
            "ballast_water": ballast_water_impact,
            "marine_life_risk": marine_life_risk,
            "overall_environmental_score": (carbon_score + energy_score) / 2,
        }

    def calculate_social_impact(self, shipment: Shipment,
                                mode: str) -> Dict[str, float]:
        """Calculate illustrative labor and community indicators for a mode."""
        if mode == "ocean":
            labor_standards = 0.8
            job_creation = 0.6
            working_conditions = 0.7
        else:
            labor_standards = 0.9
            job_creation = 0.4
            working_conditions = 0.85

        destination = shipment.destination
        if destination in ["hamburg", "rotterdam"]:
            community_impact = 0.4 if mode == "ocean" else 0.2
            noise_pollution = 0.3 if mode == "ocean" else 0.7
        else:
            community_impact = 0.3 if mode == "ocean" else 0.1
            noise_pollution = 0.2 if mode == "ocean" else 0.5

        special_impact = 0.0
        if "hazardous" in shipment.special_requirements:
            special_impact = 0.3 if mode == "ocean" else 0.4
        if "temperature_controlled" in shipment.special_requirements:
            special_impact += 0.1

        overall_social_score = (
            labor_standards + job_creation + working_conditions
            + community_impact - noise_pollution
        ) / 4
        return {
            "labor_standards": labor_standards,
            "job_creation": job_creation,
            "working_conditions": working_conditions,
            "community_impact": community_impact,
            "noise_pollution": noise_pollution,
            "special_requirements": special_impact,
            "overall_social_score": overall_social_score,
        }

    def calculate_stakeholder_utilities(
        self, shipment: Shipment, mode: str,
        environmental_impact: Dict[str, float],
        social_impact: Dict[str, float], economic_cost: float,
    ) -> Dict[str, float]:
        """Calculate the implemented interest-weighted utility for each stakeholder."""
        utilities = {}
        for stakeholder in self.stakeholders:
            utility = 0.0
            for interest, weight in stakeholder.utility_weights.items():
                if interest in ("cost", "profit"):
                    component = self._normalize_impact(
                        economic_cost, self.economic_baseline, True
                    )
                elif interest in ("service", "reliability"):
                    transit_time = self.params.transit_times[mode][shipment.destination]
                    component = self._normalize_impact(transit_time, 15, True)
                elif interest in ("environment", "carbon_emissions"):
                    component = environmental_impact["overall_environmental_score"]
                elif interest in ("labor_standards", "safety"):
                    component = social_impact["labor_standards"]
                elif interest in ("local_jobs", "community"):
                    component = social_impact["community_impact"]
                elif interest == "noise":
                    component = 1.0 - social_impact["noise_pollution"]
                elif interest == "energy_efficiency":
                    component = environmental_impact["energy_score"]
                else:
                    component = 0.0
                utility += weight * component

            importance = shipment.stakeholder_importance.get(
                stakeholder.stakeholder_type, 0.1
            )
            utilities[stakeholder.stakeholder_id] = utility * importance
        return utilities

    def calculate_overall_ethical_score(
        self, environmental_impact: Dict[str, float],
        social_impact: Dict[str, float], stakeholder_utilities: Dict[str, float],
    ) -> float:
        """Combine the stated environmental, social, and utility components."""
        environmental_score = environmental_impact["overall_environmental_score"]
        social_score = social_impact["overall_social_score"]
        mean_utility = float(np.mean(list(stakeholder_utilities.values())))
        score = (
            (self.carbon_impact_weight + self.energy_impact_weight)
            * environmental_score
            + self.social_impact_weight * social_score
            + self.economic_impact_weight * mean_utility
        )
        return max(0.0, min(1.0, score))

## 3. Check ethical constraints

A constraint checker evaluates each defined metric against its threshold and destination scope. An inapplicable destination-specific constraint is recorded as compliant for that mode evaluation; consequently, the overall compliance fraction counts all defined checks, including checks that do not apply. This convention makes the fraction easy to compute but means it should not be interpreted as an applicability-adjusted audit rate.

Hard constraints are used for feasibility screening. A soft-constraint violation incurs half of its configured penalty weight; a guidance-principle violation does not incur a penalty. The compliance summary aggregates all recorded evaluations, including evaluations of alternatives considered for transparency, so it describes the evaluation history rather than only the final selected modes.

In [ ]:
class EthicalConstraintChecker:
    """Check constraint compliance and summarize recorded mode evaluations."""

    def __init__(self, constraints: List[EthicalConstraint]):
        self.constraints = constraints
        self.constraint_map = {constraint.constraint_id: constraint
                               for constraint in constraints}
        self.compliance_history = []

    def check_compliance(self, shipment: Shipment, mode: str,
                         environmental_impact: Dict[str, float],
                         social_impact: Dict[str, float]) -> Dict[str, bool]:
        """Evaluate every constraint and record the check for audit summaries."""
        results = {
            constraint.constraint_id: self._check_single_constraint(
                constraint, shipment, mode, environmental_impact, social_impact
            )
            for constraint in self.constraints
        }
        self.compliance_history.append({
            "shipment_id": shipment.id,
            "mode": mode,
            "compliance": results,
        })
        return results

    def _check_single_constraint(
        self, constraint: EthicalConstraint, shipment: Shipment, mode: str,
        environmental_impact: Dict[str, float], social_impact: Dict[str, float],
    ) -> bool:
        """Compare the constraint's metric with its threshold when applicable."""
        if (constraint.applicable_destinations
                and shipment.destination not in constraint.applicable_destinations):
            return True

        if constraint.metric == "carbon_emissions":
            return environmental_impact["carbon_emissions"] <= constraint.threshold
        if constraint.metric == "labor_standards":
            return social_impact["labor_standards"] >= constraint.threshold
        if constraint.metric == "noise_pollution":
            noise_on_0_to_100_scale = social_impact["noise_pollution"] * 100
            return noise_on_0_to_100_scale <= constraint.threshold
        if constraint.metric == "energy_efficiency":
            energy_per_kg = environmental_impact["energy_consumption"] / shipment.weight
            return energy_per_kg <= constraint.threshold
        if constraint.metric == "community_impact":
            return social_impact["community_impact"] <= constraint.threshold
        raise ValueError(f"Unsupported constraint metric: {constraint.metric}")

    def calculate_compliance_score(self, compliance_results: Dict[str, bool]) -> float:
        """Return the fraction of recorded constraint checks that pass."""
        if not compliance_results:
            return 1.0
        return sum(compliance_results.values()) / len(compliance_results)

    def get_violation_penalty(self, compliance_results: Dict[str, bool]) -> float:
        """Apply configured penalties to failed hard and soft constraints."""
        penalty = 0.0
        for constraint_id, compliant in compliance_results.items():
            if compliant:
                continue
            constraint = self.constraint_map[constraint_id]
            if constraint.constraint_type == "hard_constraint":
                penalty += constraint.penalty_weight
            elif constraint.constraint_type == "soft_constraint":
                penalty += constraint.penalty_weight * 0.5
        return penalty

    def get_compliance_summary(self) -> Dict:
        """Summarize all recorded constraint checks and per-constraint rates."""
        if not self.compliance_history:
            return {"total_checks": 0, "compliant_checks": 0,
                    "compliance_rate": 0.0, "constraint_compliance": {}}

        total_checks = len(self.compliance_history) * len(self.constraints)
        compliant_checks = sum(
            sum(check["compliance"].values())
            for check in self.compliance_history
        )
        per_constraint = {}
        for constraint in self.constraints:
            values = [check["compliance"][constraint.constraint_id]
                      for check in self.compliance_history]
            per_constraint[constraint.constraint_id] = float(np.mean(values))

        return {
            "total_checks": total_checks,
            "compliant_checks": compliant_checks,
            "compliance_rate": compliant_checks / total_checks,
            "constraint_compliance": per_constraint,
        }

## 4. Evaluate alternatives and select a mode

For each shipment, the optimizer evaluates ocean and air independently. It first checks hard-constraint feasibility. If at least one mode is feasible, only feasible modes compete on the weighted objective. If neither mode is feasible, it chooses the mode with fewer hard violations and then compares objective scores. This fallback provides a defined decision, but the report must still show that hard constraints remain violated.

The normalized economic score is higher when cost is lower. Environmental and social scores are also higher when their modeled performance is better. Soft-constraint penalties reduce the objective. When candidate objective scores are equal within a small floating-point tolerance, the implementation uses a deterministic tie-break: ocean is considered first and retained on a tie.

The `EthicalDecision` object preserves the selected mode's measures and includes an evaluation of the other mode. Decision history and performance metrics are maintained by the optimizer. The constraint checker is owned by the optimizer, so its compliance history and the displayed summary refer to the same evaluations.

In [ ]:
class ValueAlignedOptimizer:
    """Compare ocean and air options using hard screening and a weighted score."""

    def __init__(self, params: TransportParams, stakeholders: List[Stakeholder],
                 ethical_constraints: List[EthicalConstraint]):
        self.params = params
        self.stakeholders = stakeholders
        self.stakeholder_map = {stakeholder.stakeholder_id: stakeholder
                                for stakeholder in stakeholders}
        self.ethical_constraints = ethical_constraints
        self.ethical_scorer = EthicalScoringSystem(params, stakeholders)
        self.constraint_checker = EthicalConstraintChecker(ethical_constraints)
        self.economic_weight = 0.3
        self.environmental_weight = 0.35
        self.social_weight = 0.35
        self.decision_history = []
        self.performance_metrics = {
            "economic_performance": [],
            "environmental_performance": [],
            "social_performance": [],
            "ethical_compliance": [],
            "stakeholder_satisfaction": [],
        }

    def optimize_freight_mode(self, shipment: Shipment) -> EthicalDecision:
        """Evaluate both modes, apply hard screening, and record the decision."""
        evaluations = {
            mode: self._evaluate_mode(shipment, mode)
            for mode in ("ocean", "air")
        }
        viable_options = [
            (mode, evaluation)
            for mode, evaluation in evaluations.items()
            if not self._violates_hard_constraints(evaluation["compliance"])
        ]

        if viable_options:
            chosen_mode, chosen_evaluation = self._select_best_option(viable_options)
        else:
            chosen_mode = min(
                evaluations,
                key=lambda mode: (
                    self._count_hard_violations(evaluations[mode]["compliance"]),
                    -evaluations[mode]["objective_score"],
                    mode != "ocean",
                ),
            )
            chosen_evaluation = evaluations[chosen_mode]

        decision = self._create_ethical_decision(
            shipment, chosen_mode, chosen_evaluation, evaluations
        )
        self.decision_history.append(decision)
        self._update_performance_metrics(decision)
        return decision

    def _evaluate_mode(self, shipment: Shipment, mode: str) -> Dict:
        """Calculate all modeled measures and the mode's scalar objective score."""
        economic_cost = self._calculate_total_cost(shipment, mode)
        environmental_impact = self.ethical_scorer.calculate_environmental_impact(
            shipment, mode
        )
        social_impact = self.ethical_scorer.calculate_social_impact(shipment, mode)
        stakeholder_utilities = self.ethical_scorer.calculate_stakeholder_utilities(
            shipment, mode, environmental_impact, social_impact, economic_cost
        )
        compliance = self.constraint_checker.check_compliance(
            shipment, mode, environmental_impact, social_impact
        )
        ethical_score = self.ethical_scorer.calculate_overall_ethical_score(
            environmental_impact, social_impact, stakeholder_utilities
        )
        penalty = self.constraint_checker.get_violation_penalty(compliance)
        objective_score = (
            self.economic_weight * self._normalize_cost(economic_cost)
            + self.environmental_weight
            * environmental_impact["overall_environmental_score"]
            + self.social_weight * social_impact["overall_social_score"]
            - penalty * 0.1
        )
        return {
            "mode": mode,
            "economic_cost": economic_cost,
            "environmental_impact": environmental_impact,
            "social_impact": social_impact,
            "stakeholder_utilities": stakeholder_utilities,
            "compliance": compliance,
            "ethical_score": ethical_score,
            "compliance_penalty": penalty,
            "objective_score": objective_score,
        }

    def _calculate_total_cost(self, shipment: Shipment, mode: str) -> float:
        """Add transport, early-arrival holding, lateness, and specified surcharges."""
        destination = shipment.destination
        transit_time = self.params.transit_times[mode][destination]
        transport_cost = shipment.weight * self.params.costs[mode][destination]
        holding_cost = (
            max(0, shipment.due_date - transit_time)
            * self.params.holding_rate * shipment.value
        )
        late_penalty_cost = (
            max(0, transit_time - shipment.due_date) * self.params.late_penalty
        )
        special_cost = 0.0
        if "temperature_controlled" in shipment.special_requirements:
            special_cost += shipment.weight * 0.5
        if "hazardous" in shipment.special_requirements:
            special_cost += shipment.weight * 1.2
        return transport_cost + holding_cost + late_penalty_cost + special_cost

    def _normalize_cost(self, cost: float) -> float:
        """Map cost to a bounded score where lower cost receives a higher score."""
        baseline_cost = 5000.0
        return max(0.0, min(1.0, baseline_cost / max(cost, 100.0)))

    def _violates_hard_constraints(self, compliance: Dict[str, bool]) -> bool:
        """Return whether any hard constraint fails for this mode evaluation."""
        return any(
            constraint.constraint_type == "hard_constraint" and not compliant
            for constraint_id, compliant in compliance.items()
            for constraint in [self.constraint_checker.constraint_map[constraint_id]]
        )

    def _count_hard_violations(self, compliance: Dict[str, bool]) -> int:
        """Count failed hard constraints for the infeasible-option fallback."""
        return sum(
            constraint.constraint_type == "hard_constraint" and not compliant
            for constraint_id, compliant in compliance.items()
            for constraint in [self.constraint_checker.constraint_map[constraint_id]]
        )

    def _select_best_option(self, viable_options: List[Tuple[str, Dict]]) -> Tuple[str, Dict]:
        """Choose the highest-scoring feasible option with a stable tie-break."""
        best_mode, best_evaluation = viable_options[0]
        for mode, evaluation in viable_options[1:]:
            score = evaluation["objective_score"]
            best_score = best_evaluation["objective_score"]
            if score > best_score and not math.isclose(
                score, best_score, rel_tol=1e-9, abs_tol=1e-12
            ):
                best_mode, best_evaluation = mode, evaluation
        return best_mode, best_evaluation

    def _create_ethical_decision(
        self, shipment: Shipment, mode: str, evaluation: Dict,
        all_evaluations: Dict[str, Dict],
    ) -> EthicalDecision:
        """Assemble the selected evaluation and a comparable alternative record."""
        transparency_report = self._generate_transparency_report(
            shipment, mode, evaluation
        )
        alternatives = self._generate_alternatives_analysis(
            shipment, mode, all_evaluations
        )
        return EthicalDecision(
            shipment_id=shipment.id,
            chosen_mode=mode,
            economic_cost=evaluation["economic_cost"],
            environmental_impact=evaluation["environmental_impact"],
            social_impact=evaluation["social_impact"],
            stakeholder_utilities=evaluation["stakeholder_utilities"],
            ethical_compliance=evaluation["compliance"],
            ethical_score=evaluation["ethical_score"],
            transparency_report=transparency_report,
            alternative_considered=alternatives,
        )

    def _generate_transparency_report(self, shipment: Shipment, mode: str,
                                      evaluation: Dict) -> str:
        """Explain the selected mode, key measures, and constraint outcomes."""
        lines = [
            f"Decision for shipment {shipment.id} to {shipment.destination}:",
            f"Chosen mode: {mode.upper()}",
            f"Economic cost: ${evaluation['economic_cost']:,.0f}",
            "Environmental score: "
            f"{evaluation['environmental_impact']['overall_environmental_score']:.1%}",
            "Social score: "
            f"{evaluation['social_impact']['overall_social_score']:.1%}",
            "Ethical score: "
            f"{evaluation['ethical_score']:.1%}",
            "Constraint-check fraction: "
            f"{self.constraint_checker.calculate_compliance_score(evaluation['compliance']):.1%}",
            "\nConstraint status:",
        ]
        for constraint_id, compliant in evaluation["compliance"].items():
            constraint = self.constraint_checker.constraint_map[constraint_id]
            status = "PASS" if compliant else "FAIL"
            lines.append(f"- {constraint.name}: {status}")
        lines.append("\nStakeholder utility indicators:")
        for stakeholder_id, utility in evaluation["stakeholder_utilities"].items():
            stakeholder = self.stakeholder_map[stakeholder_id]
            lines.append(f"- {stakeholder.name}: {utility:.3f}")
        if self._violates_hard_constraints(evaluation["compliance"]):
            lines.append(
                "\nNo hard-feasible mode was available; this selection is the "
                "least-violating fallback, not a compliant solution."
            )
        return "\n".join(lines)

    def _generate_alternatives_analysis(
        self, shipment: Shipment, chosen_mode: str,
        evaluations: Dict[str, Dict],
    ) -> List[Dict]:
        """Report the already-evaluated non-selected mode without rechecking it."""
        alternatives = []
        for mode, evaluation in evaluations.items():
            if mode == chosen_mode:
                continue
            hard_violations = self._count_hard_violations(evaluation["compliance"])
            if hard_violations:
                reason = f"Violates {hard_violations} hard constraint(s)"
            else:
                reason = "Lower objective score among feasible modes"
            alternatives.append({
                "mode": mode,
                "economic_cost": evaluation["economic_cost"],
                "environmental_score": evaluation["environmental_impact"]
                ["overall_environmental_score"],
                "social_score": evaluation["social_impact"]["overall_social_score"],
                "ethical_compliance": self.constraint_checker.calculate_compliance_score(
                    evaluation["compliance"]
                ),
                "reason_for_rejection": reason,
            })
        return alternatives

    def _update_performance_metrics(self, decision: EthicalDecision) -> None:
        """Append the selected decision's measures to the reporting histories."""
        self.performance_metrics["economic_performance"].append(
            self._normalize_cost(decision.economic_cost)
        )
        self.performance_metrics["environmental_performance"].append(
            decision.environmental_impact["overall_environmental_score"]
        )
        self.performance_metrics["social_performance"].append(
            decision.social_impact["overall_social_score"]
        )
        self.performance_metrics["ethical_compliance"].append(
            self.constraint_checker.calculate_compliance_score(
                decision.ethical_compliance
            )
        )
        self.performance_metrics["stakeholder_satisfaction"].append(
            float(np.mean(list(decision.stakeholder_utilities.values())))
        )

    def get_performance_summary(self) -> Dict:
        """Return means over selected shipment decisions and evaluation history."""
        if not self.decision_history:
            return {"error": "No decisions have been recorded."}
        return {
            "total_decisions": len(self.decision_history),
            "average_economic_performance": float(np.mean(
                self.performance_metrics["economic_performance"])),
            "average_environmental_performance": float(np.mean(
                self.performance_metrics["environmental_performance"])),
            "average_social_performance": float(np.mean(
                self.performance_metrics["social_performance"])),
            "average_ethical_compliance": float(np.mean(
                self.performance_metrics["ethical_compliance"])),
            "average_stakeholder_satisfaction": float(np.mean(
                self.performance_metrics["stakeholder_satisfaction"])),
            "compliance_summary": self.constraint_checker.get_compliance_summary(),
        }


optimizer = ValueAlignedOptimizer(params, stakeholders, ethical_constraints)
constraint_checker = optimizer.constraint_checker
print("Value-aligned optimizer initialized:")
print(f"  Economic objective weight: {optimizer.economic_weight:.0%}")
print(f"  Environmental objective weight: {optimizer.environmental_weight:.0%}")
print(f"  Social objective weight: {optimizer.social_weight:.0%}")
print(f"  Ethical constraints: {len(optimizer.ethical_constraints)}")

## 5. Run the decisions and inspect the reports

The next cell applies the same decision sequence to each shipment, prints the selected mode and its key measures, and displays aggregate metrics. The detailed report is for the fourth shipment in the input list; it is selected by position for illustration, while its shipment identifier is printed explicitly.

In [ ]:
ethical_decisions = []
print("Starting value-aligned freight-mode evaluation")
print("=" * 72)

for shipment in shipments:
    decision = optimizer.optimize_freight_mode(shipment)
    ethical_decisions.append(decision)
    violations = [
        constraint_id for constraint_id, compliant
        in decision.ethical_compliance.items() if not compliant
    ]
    print(f"\nShipment {shipment.id}: {shipment.destination}, {shipment.priority} priority")
    print(f"  Selected mode: {decision.chosen_mode.upper()}")
    print(f"  Economic cost: ${decision.economic_cost:,.0f}")
    print(f"  Ethical score: {decision.ethical_score:.1%}")
    print("  Constraint-check fraction: "
          f"{constraint_checker.calculate_compliance_score(decision.ethical_compliance):.1%}")
    print(f"  Failed checks: {violations if violations else 'None'}")
    print("  Carbon emissions: "
          f"{decision.environmental_impact['carbon_emissions']:.1f} kg CO2")
    print("  Social score: "
          f"{decision.social_impact['overall_social_score']:.1%}")

performance_summary = optimizer.get_performance_summary()
compliance_summary = constraint_checker.get_compliance_summary()
print("\nSelected-decision performance summary")
print("=" * 72)
print(f"Number of shipment decisions: {performance_summary['total_decisions']}")
print("Mean normalized economic performance: "
      f"{performance_summary['average_economic_performance']:.1%}")
print("Mean environmental score: "
      f"{performance_summary['average_environmental_performance']:.1%}")
print("Mean social score: "
      f"{performance_summary['average_social_performance']:.1%}")
print("Mean selected-mode compliance fraction: "
      f"{performance_summary['average_ethical_compliance']:.1%}")
print("Mean stakeholder utility indicator: "
      f"{performance_summary['average_stakeholder_satisfaction']:.3f}")

print("\nConstraint-check history across evaluated modes")
print(f"Total individual checks: {compliance_summary['total_checks']}")
print(f"Passing checks: {compliance_summary['compliant_checks']}")
print(f"Aggregate check pass fraction: {compliance_summary['compliance_rate']:.1%}")
for constraint_id, rate in compliance_summary["constraint_compliance"].items():
    print(f"  {constraint_checker.constraint_map[constraint_id].name}: {rate:.1%}")

complex_decision = ethical_decisions[3]
print(f"\nTransparency report for shipment {complex_decision.shipment_id}")
print("=" * 72)
print(complex_decision.transparency_report)

## 6. Visualize performance and environmental comparisons

The panels summarize selected-decision performance, constraint pass fractions by constraint class, stakeholder utility indicators, and the carbon-emissions comparison between both modes for each shipment. In the carbon plot, each point represents one shipment: the horizontal coordinate is its ocean emissions and the vertical coordinate is its air emissions. Color and marker shape identify the selected mode. These are comparisons of modeled alternatives, not measured emissions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Value-Aligned Freight Decision Summary", fontsize=16, fontweight="bold")

# A compact profile of the selected decisions' normalized summary measures.
ax = axes[0, 0]
performance_labels = ["Economic", "Environmental", "Social", "Compliance"]
performance_values = [
    performance_summary["average_economic_performance"],
    performance_summary["average_environmental_performance"],
    performance_summary["average_social_performance"],
    performance_summary["average_ethical_compliance"],
]
ax.bar(performance_labels, performance_values, color=["steelblue", "seagreen",
       "darkorange", "slateblue"])
ax.set_ylim(0, 1)
ax.set_ylabel("Mean score or pass fraction")
ax.set_title("Selected-Decision Summary")
ax.grid(axis="y", alpha=0.3)

# Averages include all recorded mode evaluations; inapplicable checks pass by convention.
ax = axes[0, 1]
constraint_types = ["Hard", "Soft", "Guidance"]
compliance_by_type = {name: [] for name in constraint_types}
for constraint in ethical_constraints:
    if constraint.constraint_type == "hard_constraint":
        group = "Hard"
    elif constraint.constraint_type == "soft_constraint":
        group = "Soft"
    else:
        group = "Guidance"
    compliance_by_type[group].append(
        compliance_summary["constraint_compliance"][constraint.constraint_id]
    )
mean_by_type = [
    float(np.mean(compliance_by_type[group])) if compliance_by_type[group] else 0.0
    for group in constraint_types
]
bars = ax.bar(constraint_types, mean_by_type,
              color=["indianred", "darkorange", "seagreen"])
ax.set_ylim(0, 1.12)
ax.set_xlabel("Constraint class")
ax.set_ylabel("Mean pass fraction")
ax.set_title("Constraint Checks by Class")
ax.grid(axis="y", alpha=0.3)
for bar, value in zip(bars, mean_by_type):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.02,
            f"{value:.1%}", ha="center", va="bottom")

# Each stakeholder's values are the utility indicators across selected decisions.
ax = axes[1, 0]
stakeholder_values = [
    [decision.stakeholder_utilities[stakeholder.stakeholder_id]
     for decision in ethical_decisions]
    for stakeholder in stakeholders
]
ax.boxplot(stakeholder_values, labels=[s.name for s in stakeholders],
           patch_artist=True)
ax.set_xlabel("Stakeholder group")
ax.set_ylabel("Utility indicator")
ax.set_title("Stakeholder Utility Indicators")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.3)

# Compare the two modeled carbon outcomes for every shipment.
ax = axes[1, 1]
ocean_carbon = []
air_carbon = []
for shipment in shipments:
    ocean_carbon.append(
        shipment.weight * params.carbon_emissions["ocean"][shipment.destination]
    )
    air_carbon.append(
        shipment.weight * params.carbon_emissions["air"][shipment.destination]
    )
for index, decision in enumerate(ethical_decisions):
    if decision.chosen_mode == "ocean":
        ax.scatter(ocean_carbon[index], air_carbon[index], color="navy",
                   marker="o", alpha=0.75, label="Ocean selected" if index == 0 else None)
    else:
        ax.scatter(ocean_carbon[index], air_carbon[index], color="firebrick",
                   marker="^", alpha=0.75, label="Air selected" if index == 0 else None)
max_emissions = max(ocean_carbon + air_carbon)
ax.plot([0, max_emissions], [0, max_emissions], "k--", label="Equal emissions")
ax.set_xlabel("Ocean emissions (kg CO2)")
ax.set_ylabel("Air emissions (kg CO2)")
ax.set_title("Mode Emissions by Shipment")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Examine stakeholder indicators and trade-offs

The following analysis groups utility indicators by stakeholder type, calculates descriptive correlations among selected-decision measures, and displays a two-objective dominance view. A point is called non-dominated in the plotted sample if no other selected shipment has both a higher normalized economic score and a higher environmental score, with at least one strict improvement. Because points correspond to different shipments rather than alternative modes for the same shipment, this is a descriptive cross-shipment comparison—not the optimizer's feasible Pareto frontier.

Correlations can be undefined when a measure has no variation. The code checks for that case rather than presenting an undefined number as a meaningful result. The cost-versus-ethical-score plot uses a fitted line only when the data contain sufficient variation.

In [ ]:
def safe_correlation(first: List[float], second: List[float]) -> float:
    """Return a Pearson correlation, or NaN when either series is constant."""
    if len(first) < 2 or len(second) < 2:
        return float("nan")
    if np.isclose(np.std(first), 0.0) or np.isclose(np.std(second), 0.0):
        return float("nan")
    return float(np.corrcoef(first, second)[0, 1])


def demonstrate_stakeholder_analysis(
    ethical_decisions: List[EthicalDecision], stakeholders: List[Stakeholder],
) -> None:
    """Print stakeholder utility summaries and descriptive objective correlations."""
    print("Stakeholder analysis and trade-offs")
    print("=" * 72)
    for stakeholder in stakeholders:
        utilities = [
            decision.stakeholder_utilities[stakeholder.stakeholder_id]
            for decision in ethical_decisions
        ]
        print(f"{stakeholder.stakeholder_type.title()}: "
              f"mean {np.mean(utilities):.3f}, standard deviation {np.std(utilities):.3f}")

    economic_scores = [optimizer._normalize_cost(d.economic_cost)
                       for d in ethical_decisions]
    environmental_scores = [d.environmental_impact["overall_environmental_score"]
                            for d in ethical_decisions]
    social_scores = [d.social_impact["overall_social_score"]
                     for d in ethical_decisions]
    print("\nDescriptive correlations across selected shipment decisions:")
    print("  Economic and environmental: "
          f"{safe_correlation(economic_scores, environmental_scores):.3f}")
    print("  Social and environmental: "
          f"{safe_correlation(social_scores, environmental_scores):.3f}")

    threshold = 0.3
    print(f"\nDecisions with a score range greater than {threshold:.0%}:")
    for decision, economic, environmental, social in zip(
        ethical_decisions, economic_scores, environmental_scores, social_scores
    ):
        scores = {"Economic": economic, "Environmental": environmental,
                  "Social": social}
        score_range = max(scores.values()) - min(scores.values())
        if score_range > threshold:
            lowest = min(scores, key=scores.get)
            print(f"  Shipment {decision.shipment_id} ({decision.chosen_mode.upper()}): "
                  f"economic {economic:.1%}, environmental {environmental:.1%}, "
                  f"social {social:.1%}; lowest indicator: {lowest.lower()}")


demonstrate_stakeholder_analysis(ethical_decisions, stakeholders)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Stakeholder Indicators and Objective Comparisons",
             fontsize=16, fontweight="bold")

# Heatmap rows are stakeholder groups and columns are selected shipment decisions.
ax = axes[0, 0]
stakeholder_matrix = np.array([
    [decision.stakeholder_utilities[stakeholder.stakeholder_id]
     for decision in ethical_decisions]
    for stakeholder in stakeholders
])
image = ax.imshow(stakeholder_matrix, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(ethical_decisions)))
ax.set_xticklabels([f"S{d.shipment_id}" for d in ethical_decisions], rotation=45)
ax.set_yticks(range(len(stakeholders)))
ax.set_yticklabels([s.name for s in stakeholders])
ax.set_xlabel("Shipment")
ax.set_ylabel("Stakeholder group")
ax.set_title("Stakeholder Utility Indicators")
fig.colorbar(image, ax=ax, label="Utility indicator")

# Correlation matrix is descriptive and handles constant series as undefined.
ax = axes[0, 1]
compliance_scores = [constraint_checker.calculate_compliance_score(d.ethical_compliance)
                     for d in ethical_decisions]
objective_names = ["Economic", "Environmental", "Social", "Compliance"]
objective_series = [economic_scores, environmental_scores, social_scores,
                    compliance_scores]
correlation_matrix = np.array([
    [safe_correlation(first, second) for second in objective_series]
    for first in objective_series
])
image = ax.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(objective_names)))
ax.set_xticklabels(objective_names, rotation=35)
ax.set_yticks(range(len(objective_names)))
ax.set_yticklabels(objective_names)
ax.set_title("Objective Correlations")
fig.colorbar(image, ax=ax, label="Pearson correlation")
for row in range(len(objective_names)):
    for column in range(len(objective_names)):
        value = correlation_matrix[row, column]
        label = f"{value:.2f}" if np.isfinite(value) else "N/A"
        ax.text(column, row, label, ha="center", va="center", color="black")

# Mark sample points not dominated by another selected shipment.
ax = axes[1, 0]
ax.scatter(economic_scores, environmental_scores, color="steelblue",
           alpha=0.75, label="Selected shipment")
non_dominated = []
for index, (economic, environmental) in enumerate(
    zip(economic_scores, environmental_scores)
):
    dominated = any(
        other_index != index
        and other_economic >= economic
        and other_environmental >= environmental
        and (other_economic > economic or other_environmental > environmental)
        for other_index, (other_economic, other_environmental)
        in enumerate(zip(economic_scores, environmental_scores))
    )
    if not dominated:
        non_dominated.append(index)
ax.scatter([economic_scores[i] for i in non_dominated],
           [environmental_scores[i] for i in non_dominated],
           color="darkorange", marker="*", s=150, edgecolors="black",
           label="Non-dominated in this sample")
ax.set_xlabel("Normalized economic performance")
ax.set_ylabel("Environmental score")
ax.set_title("Cross-Shipment Objective Comparison")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)

# A descriptive linear trend is omitted if a fitted line is not identifiable.
ax = axes[1, 1]
selected_costs = [decision.economic_cost for decision in ethical_decisions]
ethical_scores = [decision.ethical_score for decision in ethical_decisions]
for mode, color in (("ocean", "navy"), ("air", "firebrick")):
    indices = [i for i, decision in enumerate(ethical_decisions)
               if decision.chosen_mode == mode]
    ax.scatter([selected_costs[i] for i in indices],
               [ethical_scores[i] for i in indices], color=color, alpha=0.75,
               label=f"{mode.title()} selected")
if len(set(selected_costs)) > 1:
    coefficients = np.polyfit(selected_costs, ethical_scores, 1)
    x_values = np.linspace(min(selected_costs), max(selected_costs), 100)
    ax.plot(x_values, np.polyval(coefficients, x_values), "g--",
            label="Descriptive linear fit")
ax.set_xlabel("Economic cost (USD)")
ax.set_ylabel("Ethical score")
ax.set_title("Cost and Ethical-Score Indicators")
ax.legend()
ax.grid(alpha=0.3)
correlation = safe_correlation(selected_costs, ethical_scores)
correlation_text = f"Cost-score correlation: {correlation:.3f}"
if not np.isfinite(correlation):
    correlation_text = "Cost-score correlation: undefined"
ax.text(0.04, 0.96, correlation_text, transform=ax.transAxes,
        va="top", bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.8})

plt.tight_layout()
plt.show()

## 8. Interpretation, strengths, and limitations

### What this framework contributes

- **Explicit value dimensions:** Economic, environmental, and social measures appear separately in the mode score.
- **Hard-constraint screening:** A mode that violates a hard constraint is excluded whenever at least one hard-feasible alternative exists.
- **Transparent comparisons:** The selected mode, compliance checks, stakeholder indicators, and alternative-mode measures are available for inspection.
- **Consistent evaluation sequence:** Both modes are evaluated using the same functions and shipment data before a choice is made.

### What the model does not establish

- The weights and impact baselines encode modeling choices; the resulting rankings depend on them.
- A weighted score combines different objectives and can conceal trade-offs. The score is not a moral fact or a complete representation of stakeholder preferences.
- Several stakeholder interests have no corresponding measured input or implemented utility mapping. Those interests therefore contribute zero to the utility calculation. Stakeholder influence and veto-power fields are descriptive metadata and do not alter the selection rule.
- The illustrative labor and social values are not carrier-specific evidence. The carbon and energy calculations use supplied intensities and omit broader life-cycle boundaries.
- Since each shipment is optimized independently, the model cannot enforce aggregate capacity, coordinate shared resources, or represent interactions among decisions. Air capacity is retained as a parameter but is not part of the implemented constraints.
- The fallback in which every mode violates a hard constraint is a least-violating decision rule, not proof of compliance. Such a case requires explicit review.

### Appropriate use

This small example is useful for learning how a decision pipeline can combine a weighted objective, hard feasibility checks, soft penalties, and explanatory reporting. A real freight decision would require validated carrier and emissions data, applicable legal and labor standards, explicit capacity and service constraints, stakeholder review, and governance for infeasible or disputed decisions. Extensions can add validated constraints or richer input data while retaining the same evaluation-and-selection sequence.